In [5]:
import polars as pl

regalias_departamento_sucre = pl.read_excel("Regalias/MatrizSeguimientoEvaluacion_20260312_1822.xlsx",table_name="MatrizSeguimientoEvaluacion")
regalias_descentralizadas = pl.read_excel("Regalias/MatrizSeguimientoEvaluacion_20260312_1822.xlsx",table_name="OtrosEjecutoresDescentralizadas")

def hito_1(estado_proyecto,fecha_aprobacion,fecha_corte_gesproy):
    condicion_estado = ( pl.col(estado_proyecto) == "SIN CONTRATAR" ) | ( pl.col(estado_proyecto) == "" ) | ( pl.col(estado_proyecto).is_null() )
    condicion_fechas = (
        (~pl.col(fecha_aprobacion).is_null()) &
        (~pl.col(fecha_corte_gesproy).is_null())
    )
    return (
        pl.when(condicion_estado & condicion_fechas)
        .then(
            ( ( pl.col(fecha_corte_gesproy)-pl.col(fecha_aprobacion) ).dt.total_days())
        )
        .otherwise(None)
    )

def hito_2(estado_proyecto,fecha_corte_gesproy,fecha_primer_proceso):
    condicion_estado = ( pl.col(estado_proyecto) == "SIN CONTRATAR" ) | ( pl.col(estado_proyecto) == "" ) | ( pl.col(estado_proyecto).is_null() )
    condicion_fechas = (
        ~pl.col(fecha_primer_proceso).is_null()
    )
    return (
        pl.when(condicion_estado & condicion_fechas)
        .then(
            ( ( pl.col(fecha_corte_gesproy)-pl.col(fecha_primer_proceso) ).dt.total_days() )
        )
        .otherwise(None)
    )

def hito_3(estado_proyecto,fecha_corte_gesproy,fecha_suscripcion):
    condicion_estado = pl.col(estado_proyecto) == "CONTRATADO SIN ACTA DE INICIO"
    condicion_fecha = ~pl.col(fecha_suscripcion).is_null()
    return (
        pl.when(condicion_estado & condicion_fecha)
        .then(
            ( ( pl.col(fecha_corte_gesproy)-pl.col(fecha_suscripcion) ).dt.total_days() )
        )
        .otherwise(None)
    )

def hito_4(estado_proyecto,cpi,spi,horizonte_proyecto,fecha_corte_gesproy):
    condicion_estado = pl.col(estado_proyecto) == "CONTRATADO EN EJECUCIÓN"
    condicion_cpi_spi = ( pl.col(cpi) == 0 ) & ( pl.col(spi) == 0 )
    condicion_horizonte = pl.col(horizonte_proyecto) <= pl.col(fecha_corte_gesproy)
    return (
        pl.when(condicion_estado & condicion_cpi_spi & condicion_horizonte )
        .then(
            ( ( pl.col(fecha_corte_gesproy)-pl.col(horizonte_proyecto) ).dt.total_days() )
        )
        .otherwise(None)
    )

def hito_5(fecha_finalizacion,fecha_corte_gesproy):
    condicion_fecha = ~pl.col(fecha_finalizacion).is_null()
    return (
        pl.when(condicion_fecha)
        .then(
            ( ( pl.col(fecha_corte_gesproy)-pl.col(fecha_finalizacion) ).dt.total_days() )
        )
        .otherwise(None)
    )


def clasi_hito_1(hito_1):
    condicion = ~pl.col(hito_1).is_null()
    intervalo_1 = ( pl.col(hito_1) >= 0 ) & ( pl.col(hito_1) <= 100 )
    intervalo_2 = ( pl.col(hito_1) > 100 ) & ( pl.col(hito_1) <= 150 )
    intervalo_3 = ( pl.col(hito_1) > 150 ) & ( pl.col(hito_1) <= 180 )
    return (
        pl.when(condicion)
        .then(
            pl.when( intervalo_1 )
            .then(pl.lit("0-100"))
            .when( intervalo_2 )
            .then(pl.lit("101-150"))
            .when( intervalo_3 )
            .then(pl.lit("151-180"))
            .otherwise(pl.lit(">180"))
        )
        .otherwise(None)
    )


def clasi_hito_2(hito_2):
    condicion = ~pl.col(hito_2).is_null()
    intervalo_1 = ( pl.col(hito_2) >= 0 ) & ( pl.col(hito_2) <= 100 )
    intervalo_2 = ( pl.col(hito_2) > 100 ) & ( pl.col(hito_2) <= 150 )
    intervalo_3 = ( pl.col(hito_2) > 150 ) & ( pl.col(hito_2) <= 180 )
    return (
        pl.when(condicion)
        .then(
            pl.when( intervalo_1 )
            .then(pl.lit("0-100"))
            .when( intervalo_2 )
            .then(pl.lit("101-150"))
            .when( intervalo_3 )
            .then(pl.lit("151-180"))
            .otherwise(pl.lit(">180"))
        )
        .otherwise(None)
    )

def clasi_hito_3(hito_3):
    condicion = ~pl.col(hito_3).is_null()
    intervalo_1 = ( pl.col(hito_3) >= 0 ) & ( pl.col(hito_3) <= 30 )
    intervalo_2 = ( pl.col(hito_3) > 30 ) & ( pl.col(hito_3) <= 45 )
    intervalo_3 = ( pl.col(hito_3) > 45 ) & ( pl.col(hito_3) <= 60 )
    return (
        pl.when(condicion)
        .then(
            pl.when( intervalo_1 )
            .then(pl.lit("0-30"))
            .when( intervalo_2 )
            .then(pl.lit("31-45"))
            .when( intervalo_3 )
            .then(pl.lit("46-60"))
            .otherwise(pl.lit(">60"))
        )
        .otherwise(None)
    )

def clasi_hito_4(estado_proyecto,hito_4):
    estado = pl.col(estado_proyecto) == "CONTRATADO EN EJECUCIÓN"
    condicion = ~pl.col(hito_4).is_null()
    intervalo_1 = ( pl.col(hito_4).is_null() ) & ( pl.col(hito_4) <= 1 )
    intervalo_2 = ( pl.col(hito_4) > 1 ) & ( pl.col(hito_4) <= 3 )
    intervalo_3 = ( pl.col(hito_4) > 3 ) & ( pl.col(hito_4) <= 6 )
    return (
        pl.when(condicion)
        .then(
            pl.when( intervalo_1 )
            .then(pl.lit("0-1"))
            .when( intervalo_2 )
            .then(pl.lit("1.1-3"))
            .when( intervalo_3 )
            .then(pl.lit("3.1-6"))
            .otherwise(pl.lit(">6"))
        )
        .otherwise(None)
    )

def clasi_hito_5(hito_5):
    condicion = ~pl.col(hito_5).is_null()
    intervalo_1 = ( pl.col(hito_5) >= 0 ) & ( pl.col(hito_5) <= 100 )
    intervalo_2 = ( pl.col(hito_5) > 100 ) & ( pl.col(hito_5) <= 150 )
    intervalo_3 = ( pl.col(hito_5) > 150 ) & ( pl.col(hito_5) <= 180 )
    return (
        pl.when(condicion)
        .then(
            pl.when( intervalo_1 )
            .then(pl.lit("0-100"))
            .when( intervalo_2 )
            .then(pl.lit("101-150"))
            .when( intervalo_3 )
            .then(pl.lit("151-180"))
            .otherwise(pl.lit(">180"))
        )
        .otherwise(None)
    )
                

regalias_hitos = (
    regalias_departamento_sucre
    .select("ENTIDAD O SECRETARIA","BPIN" ,"NOMBRE PROYECTO", "ESTADO PROYECTO", "ESTADO CONTRATO",
            "CPI", "SPI", "FECHA APROBACIÓN PROYECTO", "FECHA DE APERTURA DEL PRIMER PROCESO", 
            "FECHA SUSCRIPCION", "FECHA ACTA INICIO", "HORIZONTE DEL PROYECTO", "FECHA DE FINALIZACIÓN", "FECHA DE CORTE GESPROY")
    .with_columns(pl.col("FECHA APROBACIÓN PROYECTO", "FECHA DE APERTURA DEL PRIMER PROCESO", 
            "FECHA SUSCRIPCION", "FECHA ACTA INICIO", "HORIZONTE DEL PROYECTO", "FECHA DE FINALIZACIÓN", "FECHA DE CORTE GESPROY").cast(pl.Date,strict=False))
    .with_columns(
        hito_1("ESTADO PROYECTO","FECHA APROBACIÓN PROYECTO","FECHA DE CORTE GESPROY").alias("Proyectos sin contratar sin apertura (hito 1)"),
        hito_2("ESTADO PROYECTO", "FECHA DE CORTE GESPROY","FECHA DE APERTURA DEL PRIMER PROCESO").alias("Proyectos sin contratar con apertura (hito 2)"),
        hito_3("ESTADO PROYECTO","FECHA DE CORTE GESPROY","FECHA SUSCRIPCION").alias("Proyectos contratados sin acta de inicio (hito 3)"),
        hito_4("ESTADO PROYECTO","CPI", "SPI","HORIZONTE DEL PROYECTO", "FECHA DE CORTE GESPROY").alias("Proyectos en ejecución (hito 4)"),
        hito_5("FECHA DE FINALIZACIÓN", "FECHA DE CORTE GESPROY").alias("Proyectos terminados (hito 5)")
    )
    .with_columns(
        pl.when(pl.col("ESTADO CONTRATO").str.strip_chars().str.to_uppercase() == "SUSPENDIDO")
        .then(pl.lit(1)).alias("Suspendidos"),
        pl.when(pl.col("ESTADO PROYECTO") == "PARA CIERRE")
        .then(pl.lit(1)).alias("Para cierre"),
        clasi_hito_1("Proyectos sin contratar sin apertura (hito 1)").alias("Clasificacion hito 1"),
        clasi_hito_2("Proyectos sin contratar con apertura (hito 2)").alias("Clasificacion hito 2"),
        clasi_hito_3("Proyectos contratados sin acta de inicio (hito 3)").alias("Clasificacion hito 3"),
        clasi_hito_4("ESTADO PROYECTO","Proyectos en ejecución (hito 4)").alias("Clasificacion hito 4"),
        clasi_hito_5("Proyectos terminados (hito 5)").alias("Clasificacion hito 5")
    )
)

agrupacion = (
    regalias_hitos
    .group_by("ENTIDAD O SECRETARIA")
    .agg(
        pl.col("Proyectos sin contratar sin apertura (hito 1)").mean(),
        pl.col("Proyectos sin contratar con apertura (hito 2)").mean(),
        pl.col("Proyectos contratados sin acta de inicio (hito 3)").mean(),
        pl.col("Proyectos en ejecución (hito 4)").mean(),
        pl.col("Suspendidos").sum(),
        pl.col("Proyectos terminados (hito 5)").mean(),
        pl.col("Para cierre").sum(),
    )
)

semaforos = {
    "hito_1": {
        "0-100": {
            "color": "Verde",
            "mensaje": "Proyecto se encuentra dentro de los tiempos para su primer apertura del proceso de contratación"
        },
        "101-150": {
            "color": "Naranja",
            "mensaje": "Proyecto en alerta naranja ya que presenta más de 100 días sin apertura del primer proceso precontractual"
        },
        "151-180": {
            "color": "Rojo",
            "mensaje": "Proyecto en alerta roja, más de 150 días sin apertura del primer proceso precontractual"
        },
        ">180": {
            "color": "Negro",
            "mensaje": "Proyecto en alerta negra presenta más de 180 días sin apertura del primer proceso precontractual"
        },
    },

    "hito_2": {
        "0-100": {
            "color": "Verde",
            "mensaje": "Proyecto se encuentra dentro de los tiempos para la firma del primer contrato"
        },
        "101-150": {
            "color": "Naranja",
            "mensaje": "Proyecto en alerta naranja ya que presenta más de 100 días sin firma del primer contrato"
        },
        "151-180": {
            "color": "Rojo",
            "mensaje": "Proyecto en alerta roja, más de 150 días sin firma del primer contrato"
        },
        ">180": {
            "color": "Negro",
            "mensaje": "Proyecto en alerta negra presenta más de 180 días sin firma del primer contrato"
        },
    },

    "hito_3": {
        "0-30": {
            "color": "Verde",
            "mensaje": "Proyecto se encuentra dentro de los tiempos para la firma del acta de inicio"
        },
        "31-45": {
            "color": "Naranja",
            "mensaje": "Proyecto en alerta naranja ya que presenta más de 30 días sin firma del acta de inicio"
        },
        "46-60": {
            "color": "Rojo",
            "mensaje": "Proyecto en alerta roja, más de 45 días sin firma del acta de inicio"
        },
        ">60": {
            "color": "Negro",
            "mensaje": "Proyecto en alerta negra presenta más de 60 días sin firma del acta de inicio"
        },
    },

    "hito_4": {
        "0-1": {
            "color": "Verde",
            "mensaje": "Proyecto presenta horizonte vigente"
        },
        "1.1-3": {
            "color": "Naranja",
            "mensaje": "Proyecto presenta horizonte vencido entre 1 y 3 meses"
        },
        "3.1-6": {
            "color": "Rojo",
            "mensaje": "Proyecto presenta horizonte vencido mayor a 3 meses"
        },
        ">6": {
            "color": "Negro",
            "mensaje": "Proyecto presenta horizonte vencido mayor a 6 meses"
        },
    },

    "hito_5": {
        "0-100": {
            "color": "Verde",
            "mensaje": "Proyecto se encuentra dentro de los tiempos para pasar a estado para cierre"
        },
        "101-150": {
            "color": "Naranja",
            "mensaje": "Proyecto en alerta naranja ya que presenta más de 100 días desde su terminación para pasar a estado para cierre"
        },
        "151-180": {
            "color": "Rojo",
            "mensaje": "Proyecto en alerta roja ya que presenta más de 150 días desde su terminación para pasar a estado para cierre"
        },
        ">180": {
            "color": "Negro",
            "mensaje": "Proyecto en alerta negra ya que presenta más de 180 días desde su terminación para pasar a estado para cierre"
        },
    },
}

# Evaluación Modelo Ejecutor Departamento de Sucre


# PROMEDIO CALIFICACIÓN DESEMPEÑO EN LA CONTRATACIÓN
promedio_calificacion_desempeño_en_la_contratacion = (
    regalias_departamento_sucre
    .filter(~pl.col("CALIFICACIÓN DESEMPEÑO EN LA CONTRATACIÓN").is_null())
    .select("ENTIDAD O SECRETARIA","CALIFICACIÓN DESEMPEÑO EN LA CONTRATACIÓN")
)
promedio_calificacion_desempeño_en_la_contratacion = (
    promedio_calificacion_desempeño_en_la_contratacion.group_by("ENTIDAD O SECRETARIA").agg(pl.col("CALIFICACIÓN DESEMPEÑO EN LA CONTRATACIÓN").mean())
)

# PROMEDIO CALIFICACIÓN INFORMACIÓN A TIEMPO
promedio_calificacion_informacion_a_tiempo = (
    regalias_departamento_sucre
    .filter(~pl.col("CALIFICACIÓN INFORMACIÓN A TIEMPO").is_null())
    .select("ENTIDAD O SECRETARIA","CALIFICACIÓN INFORMACIÓN A TIEMPO")
)
promedio_calificacion_informacion_a_tiempo = (
    promedio_calificacion_informacion_a_tiempo.group_by("ENTIDAD O SECRETARIA").agg(pl.col("CALIFICACIÓN INFORMACIÓN A TIEMPO").mean())
)


# PROMEDIO CALIFICACIÓN DESEMPEÑO EN LA PROGRAMACIÓN
promedio_calificacion_desempeño_en_la_programacion = (
    regalias_departamento_sucre
    .filter(~pl.col("CALIFICACIÓN EJECUCIÓN DEL PROYECTO").is_null())
    .select("ENTIDAD O SECRETARIA","CALIFICACIÓN EJECUCIÓN DEL PROYECTO")
)
promedio_calificacion_desempeño_en_la_programacion = (
    promedio_calificacion_desempeño_en_la_programacion.group_by("ENTIDAD O SECRETARIA").agg(pl.col("CALIFICACIÓN EJECUCIÓN DEL PROYECTO").mean())
)

# PROMEDIO CALIFICACIÓN CALIDAD DE LA INFORMACIÓN
promedio_calificacion_calidad_de_la_informacion = (
    regalias_departamento_sucre
    .filter(~pl.col("CALIFICACIÓN CALIDAD INFORMACIÓN").is_null())
    .select("ENTIDAD O SECRETARIA","CALIFICACIÓN CALIDAD INFORMACIÓN")
)
promedio_calificacion_calidad_de_la_informacion = (
    promedio_calificacion_calidad_de_la_informacion.group_by("ENTIDAD O SECRETARIA").agg(pl.col("CALIFICACIÓN CALIDAD INFORMACIÓN").mean())
)


# Evaluación Modelo Descentralizadas


# PROMEDIO CALIFICACIÓN DESEMPEÑO EN LA CONTRATACIÓN
promedio_calificacion_desempeño_en_la_contratacion = (
    regalias_descentralizadas
    .filter(~pl.col("CALIFICACIÓN DESEMPEÑO EN LA CONTRATACIÓN").is_null())
    .select("EJECUTOR","CALIFICACIÓN DESEMPEÑO EN LA CONTRATACIÓN")
)
promedio_calificacion_desempeño_en_la_contratacion = (
    promedio_calificacion_desempeño_en_la_contratacion.group_by("EJECUTOR").agg(pl.col("CALIFICACIÓN DESEMPEÑO EN LA CONTRATACIÓN").mean())
)

# PROMEDIO CALIFICACIÓN INFORMACIÓN A TIEMPO
promedio_calificacion_informacion_a_tiempo = (
    regalias_descentralizadas
    .filter(~pl.col("CALIFICACIÓN INFORMACIÓN A TIEMPO").is_null())
    .select("EJECUTOR","CALIFICACIÓN INFORMACIÓN A TIEMPO")
)
promedio_calificacion_informacion_a_tiempo = (
    promedio_calificacion_informacion_a_tiempo.group_by("EJECUTOR").agg(pl.col("CALIFICACIÓN INFORMACIÓN A TIEMPO").mean())
)


# PROMEDIO CALIFICACIÓN DESEMPEÑO EN LA PROGRAMACIÓN
promedio_calificacion_desempeño_en_la_programacion = (
    regalias_descentralizadas
    .filter(~pl.col("CALIFICACIÓN EJECUCIÓN DEL PROYECTO").is_null())
    .select("EJECUTOR","CALIFICACIÓN EJECUCIÓN DEL PROYECTO")
)
promedio_calificacion_desempeño_en_la_programacion = (
    promedio_calificacion_desempeño_en_la_programacion.group_by("EJECUTOR").agg(pl.col("CALIFICACIÓN EJECUCIÓN DEL PROYECTO").mean())
)

# PROMEDIO CALIFICACIÓN CALIDAD DE LA INFORMACIÓN
promedio_calificacion_calidad_de_la_informacion = (
    regalias_descentralizadas
    .filter(~pl.col("CALIFICACIÓN CALIDAD INFORMACIÓN").is_null())
    .select("EJECUTOR","CALIFICACIÓN CALIDAD INFORMACIÓN")
)
promedio_calificacion_calidad_de_la_informacion = (
    promedio_calificacion_calidad_de_la_informacion.group_by("EJECUTOR").agg(pl.col("CALIFICACIÓN CALIDAD INFORMACIÓN").mean())
)

regalias_hitos

Could not determine dtype for column 35, falling back to string
Could not determine dtype for column 47, falling back to string
Could not determine dtype for column 12, falling back to string
Could not determine dtype for column 13, falling back to string
Could not determine dtype for column 36, falling back to string


ENTIDAD O SECRETARIA,BPIN,NOMBRE PROYECTO,ESTADO PROYECTO,ESTADO CONTRATO,CPI,SPI,FECHA APROBACIÓN PROYECTO,FECHA DE APERTURA DEL PRIMER PROCESO,FECHA SUSCRIPCION,FECHA ACTA INICIO,HORIZONTE DEL PROYECTO,FECHA DE FINALIZACIÓN,FECHA DE CORTE GESPROY,Proyectos sin contratar sin apertura (hito 1),Proyectos sin contratar con apertura (hito 2),Proyectos contratados sin acta de inicio (hito 3),Proyectos en ejecución (hito 4),Proyectos terminados (hito 5),Suspendidos,Para cierre,Clasificacion hito 1,Clasificacion hito 2,Clasificacion hito 3,Clasificacion hito 4,Clasificacion hito 5
str,str,str,str,str,i64,f64,date,date,date,date,date,date,date,i64,i64,i64,i64,i64,i32,i32,str,str,str,str,str
"""Educación""","""2012000020041""","""ADECUACIÓN DE LA SEDE PARA EL …","""CONTRATADO EN EJECUCIÓN""","""Liquidado …",null,null,2012-10-19,2014-11-13,2014-11-28,2014-12-09,null,null,2026-02-15,null,null,null,null,null,null,null,null,null,null,null,null
"""Salud""","""2012000100008""","""INVESTIGACIÓN INSTITUTO INVEST…","""CONTRATADO EN EJECUCIÓN""","""Terminado …",null,null,2012-12-20,2013-11-08,2013-11-08,2014-06-13,null,null,2026-02-15,null,null,null,null,null,null,null,null,null,null,null,null
"""Desarrollo Económico""","""2013000100022""","""Implementación DE UN PROGRAMA …","""TERMINADO""","""Terminado …",null,null,2013-10-18,2014-09-25,2014-09-26,2015-02-03,null,2025-02-01,2026-02-15,null,null,null,null,379,null,null,null,null,null,null,""">180"""
"""Educación""","""2013000100074""","""Implementación PROGRAMA DE FOR…","""CONTRATADO EN EJECUCIÓN""","""En ejecución …",0,0.0,2013-07-19,2013-11-08,2013-11-08,2013-11-08,null,null,2026-02-15,null,null,null,null,null,null,null,null,null,null,null,null
"""Infraestructura""","""2016000020007""","""REHABILITACIÓN DE LA VIA SUCRE…","""TERMINADO""","""Terminado …",null,null,2016-10-14,2017-06-23,2017-09-08,2018-04-17,null,2024-08-23,2026-02-15,null,null,null,null,541,null,null,null,null,null,null,""">180"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""Energía""","""2025002700023""","""Estudio y diseño para la masif…","""SIN CONTRATAR""",null,null,null,2025-11-06,null,null,null,null,null,2026-02-15,101,null,null,null,null,null,null,"""101-150""",null,null,null,null
"""Energía""","""2025002700025""","""Cofinanciación DE SUBSIDIO AL …","""CONTRATADO SIN ACTA DE INICIO""","""Contratado sin ejecución …",null,null,2025-10-27,null,2026-02-04,null,null,null,2026-02-15,null,null,11,null,null,null,null,null,null,"""0-30""",null,null
"""Infraestructura""","""2025002700033""","""Estudios y diseños del proyect…","""SIN CONTRATAR""",null,null,null,2025-11-19,null,null,null,null,null,2026-02-15,88,null,null,null,null,null,null,"""0-100""",null,null,null,null


In [ ]:
regalias_departamento_sucre = pl.read_excel("Regalias/MatrizSeguimientoEvaluacion_20260312_1822.xlsx",table_name="MatrizSeguimientoEvaluacion")

regalias_departamento_sucre

Could not determine dtype for column 35, falling back to string
Could not determine dtype for column 47, falling back to string


In [2]:
def leer_excel_regalias(ruta: str) -> pl.DataFrame:
    df = pl.read_excel(ruta, has_header=False, infer_schema_length=0)
    encabezados = dict(zip(df.columns, df.row(1)))
    return (
        df
        .rename(encabezados)
        .slice(2,)
        .select(pl.all().name.map(lambda x: x.strip()))
    )


columnas_excluidas = ("CONTRATO NUMERO","NOMBRE CONTRATISTA","TIPO CONTRATISTA","FECHA CREACIÓN CONTRATO""VALOR CONTRATO EN EL PROYECTO","SUPERVISOR")

contratos_proyectos = (
    leer_excel_regalias("Regalias/ReportesGesproyHistorico/CG-cttos_04_marzo_20260304.xlsx")
    .select("BPIN","NO. PROCESO PRECONTRACTUAL","MODALIDAD CONTRATACION","TIPO CONTRATO",
            "CONTRATO OBJETO","CONTRATO VALOR TOTAL","ESTADO CONTRATO")
    .with_columns(pl.col("CONTRATO VALOR TOTAL").cast(pl.Float64),pl.col("BPIN","NO. PROCESO PRECONTRACTUAL","MODALIDAD CONTRATACION",
                                                                         "TIPO CONTRATO","CONTRATO OBJETO","ESTADO CONTRATO").str.strip_chars())
)

contratos_proyectos

BPIN,NO. PROCESO PRECONTRACTUAL,MODALIDAD CONTRATACION,TIPO CONTRATO,CONTRATO OBJETO,CONTRATO VALOR TOTAL,ESTADO CONTRATO
str,str,str,str,str,f64,str
"""2012000020002""","""277-2013""","""Contratación directa""","""Prestación de Servicios""","""PRESTACION DE SERVICIOS PROFES…",2.14410863e8,"""Liquidado"""
"""2012000020002""","""LP-003-2013""","""Licitación pública""","""Obra pública""","""CONSTRUCCIÓN DE 490 VIVIENDAS …",9.3634e9,"""Liquidado"""
"""2012000020002""","""SA-007-2013""","""Selección abreviada""","""Obra pública""","""MITIGACION AMBIENTAL PARA LAS …",3.7956e8,"""Liquidado"""
"""2012000020002""","""CM-002-PTS-2013""","""Concurso de méritos""","""Interventoría""","""INTERVENTORIA TÉCNICA, ADMINIS…",5.0429e8,"""Liquidado"""
"""2012000020003""","""CM-001-PTD-2013""","""Concurso de méritos""","""Consultoría""","""CONSULTORIA PARA LA ELABORACIO…",9.3271e9,"""Liquidado"""
…,…,…,…,…,…,…
"""2025000020019""","""0""","""Contratación directa""","""Prestación de Servicios""","""PRESTACION DE SERVICIOS PROFES…",5.237264e6,"""En ejecución"""
"""2025000020037""","""0""","""""","""SPGR Por definir""","""PAVIMENTACION DE LA VIA QUE CO…",7.2252e10,"""Contratado sin ejecución"""
"""2025000020037""","""CONV 023-2025""","""""","""SPGR Por definir""","""PRESTACION DE SERVICIOS DE GER…",3.7823e9,"""Contratado sin ejecución"""
